# 09 - Fine-Tune the Static 25% Pruned Model (Policy A)

**Project:** Regime-Conditional Attention Head Selection for Time Series Transformers (ReCAHS)

The project proposal's **Policy A** specifies: *"Heads with low scores will be pruned, and the model will be fine-tuned. This creates one fixed pruned model."* Experiment 04 performed the pruning step (static 25% global-importance pruning) and evaluated it **zero-shot**, without the fine-tuning step. The result was a validation improvement (0.6781 -> 0.6537, -3.60%) that did **not** generalize to the test set (0.3725 -> 0.3783, +1.55%). This notebook completes Policy A by adding the missing fine-tuning step and checking whether it closes that validation/test gap.

**Pruning mechanism.** Following notebooks 04-08, pruning is implemented as *functional* masking: the 6 lowest-importance heads (from `head_importance/summaries/static_prune_25_percent_heads.csv`) are always zeroed out via `HeadMaskController`, both during fine-tuning and at evaluation. Because a masked head's output never reaches the loss, its `query`/`key`/`value` projection weights receive zero gradient and are effectively frozen — fine-tuning only adapts the 18 active heads and the rest of the network (feed-forward layers, RevIN, output head) to compensate for the removed heads. This does **not** reduce the parameter count or FLOPs by itself (the pruned heads' weights are simply unused); a structural (hard) pruning pass that physically removes them is a separate follow-up for the efficiency analysis in a later experiment.

**What this notebook does:**
1. Load the B4 checkpoint and reproduce the zero-shot static-pruned validation/test numbers from Experiment 04 as a sanity check.
2. Fine-tune the model for a few epochs on the training set with the pruning mask permanently active, using a reduced learning rate and early stopping on masked validation MSE.
3. Re-evaluate the fine-tuned model on validation (with regime breakdown) and test, and compare against both the unpruned baseline and the zero-shot pruned model.


## 1. Mount Google Drive and import core libraries

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Define project paths

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"

PRUNING_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("HEAD_IMPORTANCE_DIR:", HEAD_IMPORTANCE_DIR)
print("PRUNING_DIR:", PRUNING_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
CHECKPOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints
HEAD_IMPORTANCE_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/head_importance
PRUNING_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments


## 3. Set up Time-Series-Library

Same reference implementation used by every other notebook in this project (see notebook 01).

In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 32.10 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.


In [5]:
%cd /content/Time-Series-Library
!pip install -q patool sktime scikit-base --no-deps
!pip install -q --no-deps einops
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

/content/Time-Series-Library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 16.5 MB/s eta 0:00:00


## 4. Download the ETTh1 dataset

In [6]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

print("Dataset exists:", tslib_data_path.exists())

Drive data not found. Downloading ETTh1...
Dataset exists: True


## 5. Load regime labels and the static pruning mask

The regime labels (for validation-side regime breakdown) and the 6 pruned (layer, head) pairs are the same ones produced in `regime_detection.ipynb` and `03_head_importance.ipynb`, already committed to the repository.

In [7]:
regime_path = REGIME_DIR / "etth1_validation_regimes_ot_seq336.csv"
regime_df = pd.read_csv(regime_path)

static_prune_path = (
    HEAD_IMPORTANCE_DIR
    / "summaries"
    / "static_prune_25_percent_heads.csv"
)
static_prune_df = pd.read_csv(static_prune_path)

print("Regime df shape:", regime_df.shape)
print("\nStatic prune heads (25%):")
display(static_prune_df[["layer", "head", "overall_importance"]])

assert len(static_prune_df) == 6, "25% pruning için 6 head bekleniyor."

Regime df shape: (2785, 13)

Static prune heads (25%):


,layer,head,overall_importance
0,0,5,-0.011536
1,1,7,-0.010845
2,2,0,-0.009892
3,1,4,-0.008351
4,1,1,-0.005822
5,0,3,-0.005073


## 6. Locate the B4 checkpoint

In [8]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob(
        "B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth"
    )
)

print("Found checkpoint candidates:")
for path in b4_checkpoint_candidates:
    print(path)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "B4 checkpoint bulunamadı. Run notebook 00 first if you don't have "
        "access to a teammate's checkpoint."
    )

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("\nSelected checkpoint:")
print(b4_checkpoint_path)

Found checkpoint candidates:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth

Selected checkpoint:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 7. Reconstruct the B4 model arguments

In [9]:
from argparse import Namespace

args = Namespace(
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    augmentation_ratio=0.0,
    lradj="type1",
    use_amp=False,

    use_gpu=torch.cuda.is_available(),
    gpu=0,
    use_multi_gpu=False,
    devices="0",
    gpu_type="cuda",
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', augmentation_ratio=0.0, lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', gpu_type='cuda', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_

## 8. Load the model and checkpoint weights

In [10]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(b4_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint)
model.eval()

print("B4 checkpoint loaded successfully.")

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
B4 checkpoint loaded successfully.


## 9. Build the train, validation and test loaders

In [11]:
from data_provider.data_factory import data_provider

train_data, train_loader = data_provider(args, flag="train")
vali_data, vali_loader = data_provider(args, flag="val")
test_data, test_loader = data_provider(args, flag="test")

print("Train dataset length:", len(train_data), "| batches:", len(train_loader))
print("Validation dataset length:", len(vali_data), "| batches:", len(vali_loader))
print("Test dataset length:", len(test_data), "| batches:", len(test_loader))

assert len(vali_data) == len(regime_df), (len(vali_data), len(regime_df))
print("Validation windows and regime labels match.")

train 8209
val 2785
test 2785
Train dataset length: 8209 | batches: 257
Validation dataset length: 2785 | batches: 88
Test dataset length: 2785 | batches: 88
Validation windows and regime labels match.


## 10. HeadMaskController

Identical to the controller used in notebooks 04-08: it monkey-patches each encoder attention layer's forward pass so a chosen subset of heads can be zeroed out, both for evaluation and — in this notebook — for training.

In [12]:
class HeadMaskController:
    def __init__(self, model):
        self.model = model
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention

            if layer_idx in self.original_forwards:
                continue

            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(
                    queries,
                    keys,
                    values,
                    attn_mask,
                    tau=None,
                    delta=None,
                ):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    queries_proj = attention_layer.query_projection(queries)
                    keys_proj = attention_layer.key_projection(keys)
                    values_proj = attention_layer.value_projection(values)

                    queries_proj = queries_proj.view(B, L, H, -1)
                    keys_proj = keys_proj.view(B, S, H, -1)
                    values_proj = values_proj.view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(
                        queries_proj,
                        keys_proj,
                        values_proj,
                        attn_mask,
                        tau=tau,
                        delta=delta,
                    )

                    if self.current_mask is not None:
                        layer_mask = self.current_mask[layer_idx].to(out.device)
                        layer_mask = layer_mask.view(1, 1, H, 1)
                        out = out * layer_mask

                    out = out.view(B, L, -1)

                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(
                layer_idx,
                attention_layer,
            )

    def remove(self):
        for layer_idx, original_forward in self.original_forwards.items():
            self.model.encoder.attn_layers[layer_idx].attention.forward = original_forward

        self.original_forwards = {}
        self.current_mask = None

    def set_all_active(self):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        self.current_mask = torch.ones(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()

    def build_static_prune_mask(self, prune_df):
        self.set_all_active()

        for _, row in prune_df.iterrows():
            layer_idx = int(row["layer"])
            head_idx = int(row["head"])
            self.current_mask[layer_idx, head_idx] = 0.0

        return self.current_mask.clone()


In [13]:
mask_controller = HeadMaskController(model)
mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads
total_heads = num_layers * num_heads

baseline_mask = torch.ones(num_layers, num_heads)
static_prune_mask = mask_controller.build_static_prune_mask(static_prune_df)

print("Total heads:", total_heads)
print("Static prune mask (0 = pruned):")
print(static_prune_mask)
print("Pruned head count:", int((static_prune_mask == 0).sum().item()))

Total heads: 24
Static prune mask (0 = pruned):
tensor([[1., 1., 1., 0., 1., 0., 1., 1.],
        [1., 0., 1., 1., 0., 1., 1., 0.],
        [0., 1., 1., 1., 1., 1., 1., 1.]])
Pruned head count: 6


## 11. Evaluation helpers

In [14]:
mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")
train_criterion = nn.MSELoss()

def compute_regime_losses_with_mask(
    model,
    loader,
    regime_df,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="validation",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            batch_size = batch_x.shape[0]
            for i in range(batch_size):
                window_id = global_index + i
                regime = regime_df.iloc[window_id]["regime"]
                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(mse=("mse", "mean"), mae=("mae", "mean"), count=("window_id", "count"))
        .reset_index()
    )

    overall = {
        "overall_mse": float(result_df["mse"].mean()),
        "overall_mae": float(result_df["mae"].mean()),
    }

    return {"overall": overall, "regime_summary": regime_summary, "window_losses": result_df}


def compute_test_metrics_with_mask(
    model,
    loader,
    mask_controller,
    mask,
    device,
    pred_len=96,
    desc="test",
):
    model.eval()
    mask_controller.set_mask(mask)

    all_mse, all_mae = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            all_mse.extend(mse_per_sample.detach().cpu().numpy().tolist())
            all_mae.extend(mae_per_sample.detach().cpu().numpy().tolist())

    return {"test_mse": float(np.mean(all_mse)), "test_mae": float(np.mean(all_mae))}


## 12. Zero-shot sanity check (reproduces Experiment 04)

Before fine-tuning anything, confirm that this notebook's baseline and static-pruned numbers match the ones already reported in `README.md` for Experiment 04 (baseline ≈ 0.6781 / 0.3725, static pruned ≈ 0.6537 / 0.3783).

In [15]:
baseline_val = compute_regime_losses_with_mask(
    model=model, loader=vali_loader, regime_df=regime_df,
    mask_controller=mask_controller, mask=baseline_mask,
    device=device, pred_len=args.pred_len, desc="Baseline validation",
)
baseline_test = compute_test_metrics_with_mask(
    model=model, loader=test_loader,
    mask_controller=mask_controller, mask=baseline_mask,
    device=device, pred_len=args.pred_len, desc="Baseline test",
)

zero_shot_pruned_val = compute_regime_losses_with_mask(
    model=model, loader=vali_loader, regime_df=regime_df,
    mask_controller=mask_controller, mask=static_prune_mask,
    device=device, pred_len=args.pred_len, desc="Zero-shot pruned validation",
)
zero_shot_pruned_test = compute_test_metrics_with_mask(
    model=model, loader=test_loader,
    mask_controller=mask_controller, mask=static_prune_mask,
    device=device, pred_len=args.pred_len, desc="Zero-shot pruned test",
)

print("Baseline val:", baseline_val["overall"], "| test:", baseline_test)
print("Zero-shot pruned val:", zero_shot_pruned_val["overall"], "| test:", zero_shot_pruned_test)

Baseline validation:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline test:   0%|          | 0/88 [00:00<?, ?it/s]

Zero-shot pruned validation:   0%|          | 0/88 [00:00<?, ?it/s]

Zero-shot pruned test:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline val: {'overall_mse': 0.678066410579095, 'overall_mae': 0.5550830315002633} | test: {'test_mse': 0.37254557146739276, 'test_mae': 0.3982142798241422}
Zero-shot pruned val: {'overall_mse': 0.6536577488778952, 'overall_mae': 0.5506856770339945} | test: {'test_mse': 0.37831396693815234, 'test_mae': 0.40442412037609726}


## 13. Fine-tune with the pruning mask permanently active

The mask stays set to `static_prune_mask` for both the forward and backward pass, so gradients for the 6 pruned heads' `query`/`key`/`value` projections are always zero (their output never reaches the loss) — only the 18 active heads and the rest of the network are updated. A small learning rate and short training budget are used since this is fine-tuning a converged checkpoint, not training from scratch; early stopping is driven by masked validation MSE, mirroring the `patience`-based early stopping used for the original B4 training run.

In [16]:
FINETUNE_LR = 1e-6
FINETUNE_MAX_EPOCHS = 10
FINETUNE_PATIENCE = 4

optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR)

mask_controller.set_mask(static_prune_mask)

best_val_mse = zero_shot_pruned_val["overall"]["overall_mse"]
best_state_dict = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

history = []

print(f"Starting point (epoch 0, zero-shot): val_mse={best_val_mse:.6f}")

for epoch in range(1, FINETUNE_MAX_EPOCHS + 1):
    model.train()
    mask_controller.set_mask(static_prune_mask)

    running_loss = 0.0
    n_batches = 0

    for batch in tqdm(train_loader, desc=f"Fine-tune epoch {epoch}"):
        batch_x, batch_y, batch_x_mark, batch_y_mark = batch

        batch_x = batch_x.float().to(device)
        batch_y = batch_y.float().to(device)
        batch_x_mark = batch_x_mark.float().to(device)
        batch_y_mark = batch_y_mark.float().to(device)

        optimizer.zero_grad()

        outputs = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
        true = batch_y[:, -args.pred_len:, :]

        loss = train_criterion(outputs, true)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1

    train_loss = running_loss / n_batches

    epoch_val = compute_regime_losses_with_mask(
        model=model, loader=vali_loader, regime_df=regime_df,
        mask_controller=mask_controller, mask=static_prune_mask,
        device=device, pred_len=args.pred_len, desc=f"Epoch {epoch} validation",
    )
    val_mse = epoch_val["overall"]["overall_mse"]
    val_mae = epoch_val["overall"]["overall_mae"]

    history.append({"epoch": epoch, "train_mse": train_loss, "val_mse": val_mse, "val_mae": val_mae})
    print(f"Epoch {epoch}: train_mse={train_loss:.6f} val_mse={val_mse:.6f} val_mae={val_mae:.6f}")

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        print(f"  -> new best (val_mse={val_mse:.6f}), checkpoint saved in memory.")
    else:
        epochs_without_improvement += 1
        print(f"  -> no improvement ({epochs_without_improvement}/{FINETUNE_PATIENCE}).")
        if epochs_without_improvement >= FINETUNE_PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state_dict)
model.eval()

history_df = pd.DataFrame(history)
display(history_df)
print("\nBest fine-tuned validation MSE:", best_val_mse)

Starting point (epoch 0, zero-shot): val_mse=0.653658


Fine-tune epoch 1:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 1 validation:   0%|          | 0/88 [00:00<?, ?it/s]

Epoch 1: train_mse=0.355627 val_mse=0.660504 val_mae=0.549345
  -> no improvement (1/4).


Fine-tune epoch 2:   0%|          | 0/257 [00:00<?, ?it/s]

...

Fine-tune epoch 4:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 4 validation:   0%|          | 0/88 [00:00<?, ?it/s]

Epoch 4: train_mse=0.351744 val_mse=0.673688 val_mae=0.553055
  -> no improvement (4/4).
Early stopping.


,epoch,train_mse,val_mse,val_mae
0,1,0.355627,0.660504,0.549345
1,2,0.353160,0.667998,0.551339
2,3,0.352062,0.672454,0.552929
3,4,0.351744,0.673688,0.553055



Best fine-tuned validation MSE: 0.6536577488778952


## 14. Evaluate the fine-tuned pruned model

In [17]:
finetuned_val = compute_regime_losses_with_mask(
    model=model, loader=vali_loader, regime_df=regime_df,
    mask_controller=mask_controller, mask=static_prune_mask,
    device=device, pred_len=args.pred_len, desc="Fine-tuned pruned validation",
)
finetuned_test = compute_test_metrics_with_mask(
    model=model, loader=test_loader,
    mask_controller=mask_controller, mask=static_prune_mask,
    device=device, pred_len=args.pred_len, desc="Fine-tuned pruned test",
)

print("Fine-tuned pruned val:", finetuned_val["overall"])
print("Fine-tuned pruned test:", finetuned_test)

display(finetuned_val["regime_summary"])

Fine-tuned pruned validation:   0%|          | 0/88 [00:00<?, ?it/s]

Fine-tuned pruned test:   0%|          | 0/88 [00:00<?, ?it/s]

Fine-tuned pruned val: {'overall_mse': 0.6536577488778952, 'overall_mae': 0.5506856770018915}
Fine-tuned pruned test: {'test_mse': 0.37831396693815234, 'test_mae': 0.40442412037609726}


,regime,mse,mae,count
0,residual,0.688016,0.563366,359
1,seasonal,0.658565,0.554353,292
2,trend,0.647206,0.548051,2134


## 15. Compare: baseline vs. zero-shot pruned vs. fine-tuned pruned

In [18]:
comparison_df = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "val_mse": baseline_val["overall"]["overall_mse"],
        "val_mae": baseline_val["overall"]["overall_mae"],
        "test_mse": baseline_test["test_mse"],
        "test_mae": baseline_test["test_mae"],
    },
    {
        "setting": "B4_static_pruning_25_zero_shot",
        "val_mse": zero_shot_pruned_val["overall"]["overall_mse"],
        "val_mae": zero_shot_pruned_val["overall"]["overall_mae"],
        "test_mse": zero_shot_pruned_test["test_mse"],
        "test_mae": zero_shot_pruned_test["test_mae"],
    },
    {
        "setting": "B4_static_pruning_25_finetuned",
        "val_mse": finetuned_val["overall"]["overall_mse"],
        "val_mae": finetuned_val["overall"]["overall_mae"],
        "test_mse": finetuned_test["test_mse"],
        "test_mae": finetuned_test["test_mae"],
    },
])

baseline_test_mse = comparison_df.loc[comparison_df["setting"] == "B4_no_pruning", "test_mse"].iloc[0]
comparison_df["test_mse_change_percent"] = (
    (comparison_df["test_mse"] - baseline_test_mse) / baseline_test_mse * 100
)

display(comparison_df)

finetuned_row = comparison_df.loc[comparison_df["setting"] == "B4_static_pruning_25_finetuned"].iloc[0]
zero_shot_row = comparison_df.loc[comparison_df["setting"] == "B4_static_pruning_25_zero_shot"].iloc[0]

print(
    f"\nFine-tuning changed test MSE from {zero_shot_row['test_mse']:.4f} "
    f"({zero_shot_row['test_mse_change_percent']:+.2f}% vs baseline) to "
    f"{finetuned_row['test_mse']:.4f} ({finetuned_row['test_mse_change_percent']:+.2f}% vs baseline)."
)

if finetuned_row["test_mse"] < baseline_test_mse:
    print("Fine-tuning closed the generalization gap: the pruned model now beats the unpruned baseline on test.")
elif finetuned_row["test_mse"] < zero_shot_row["test_mse"]:
    print("Fine-tuning narrowed the generalization gap but did not fully close it.")
else:
    print("Fine-tuning did not improve test generalization over the zero-shot pruned model.")

,setting,val_mse,val_mae,test_mse,test_mae,test_mse_change_percent
0,B4_no_pruning,0.678066,0.555083,0.372546,0.398214,0.000000
1,B4_static_pruning_25_zero_shot,0.653658,0.550686,0.378314,0.404424,1.548373
2,B4_static_pruning_25_finetuned,0.653658,0.550686,0.378314,0.404424,1.548373



Fine-tuning changed test MSE from 0.3783 (+1.55% vs baseline) to 0.3783 (+1.55% vs baseline).
Fine-tuning did not improve test generalization over the zero-shot pruned model.


## 16. Save the fine-tuned checkpoint and results

In [19]:
finetune_dir = PRUNING_DIR / "b4_static_pruning_25_finetuned"
finetune_dir.mkdir(parents=True, exist_ok=True)

torch.save(best_state_dict, finetune_dir / "checkpoint.pth")

history_df.to_csv(finetune_dir / "finetune_training_history.csv", index=False)
comparison_df.to_csv(finetune_dir / "overall_comparison.csv", index=False)
finetuned_val["regime_summary"].to_csv(finetune_dir / "finetuned_validation_regime_summary.csv", index=False)
finetuned_val["window_losses"].to_csv(finetune_dir / "finetuned_validation_window_losses.csv", index=False)

print("Saved fine-tuning outputs to:", finetune_dir)
for path in sorted(finetune_dir.iterdir()):
    print(" -", path.name)

print(
    "\nNote: this checkpoint still contains the pruned heads' original (untrained) weights. "
    "They must always be evaluated together with `static_prune_mask` (mask always active) "
    "to reproduce these numbers; loading this checkpoint without the mask is not meaningful."
)

Saved fine-tuning outputs to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_static_pruning_25_finetuned
 - checkpoint.pth
 - finetune_training_history.csv
 - finetuned_validation_regime_summary.csv
 - finetuned_validation_window_losses.csv
 - overall_comparison.csv

Note: this checkpoint still contains the pruned heads' original (untrained) weights. They must always be evaluated together with `static_prune_mask` (mask always active) to reproduce these numbers; loading this checkpoint without the mask is not meaningful.
